In [1]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
from io import StringIO

In [2]:
url = 'https://www.basketball-reference.com/leagues/NBA_2026_ratings.html'
data = requests.get(url)

soup = BeautifulSoup(data.text)
table = soup.find_all('table', id='ratings')[0]

In [3]:
links = table.find_all('a') # getting all the anchor tags/links in the table
links = [l.get("href") for l in links] # getting the urls of all the links

team_urls = [l for l in links if '/teams/' in l] # filtering to only get team urls
team_urls = [f"https://www.basketball-reference.com{l}" for l in team_urls]

In [4]:
all_teams = []

for team_url in team_urls:
    team_abb = team_url.split("/")[-2]
    
    data = requests.get(team_url)
    soup = BeautifulSoup(data.text)
    stats = soup.find_all('table', id='per_game_stats')
    
    team_data = pd.read_html(StringIO(str(stats)))[0]
    team_data.insert(3, "Team", team_abb)
    team_data = team_data.drop("Awards", axis=1)
    all_teams.append(team_data)
    time.sleep(5)

teams_df = pd.concat(all_teams)
teams_df.to_csv("stats.csv")